# 02 · Game and pitcher information (MLB Stats API and pybaseball)

Adds the non-Statcast context, then merges it onto the clean pitch table from the 'statcast_data' notebook.

| Source | What we take | Used for |
|---|---|---|
| MLB Stats API `schedule` (free, no key) | game-time temperature, conditions, venue, day/night | fatigue baseline regression (pitch count, TTO, temperature) |
| MLB Stats API `people` | birth date, throwing hand | older vs. younger pitchers (question 1) |
| `pybaseball.playerid_reverse_lookup` | names and FanGraphs / Baseball-Reference ids | joining other sources, report tables |
| Statcast training seasons (notebook 01) | fastball velocity, strikeout rate | power vs. finesse (question 1) |
| MLB Stats API `transactions` (optional, off) | injured-list stints | only if the team decides to; see ethics |

| | |
|---|---|
| **Input** | `data/clean/starts.parquet`, `data/clean/statcast_starters.parquet` |
| **Output** | `data/clean/games.parquet`, `data/clean/pitchers.parquet`, `data/clean/pitches_model.parquet` |

## Setup

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import requests
from pybaseball import playerid_reverse_lookup

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())  # repo root, wherever the notebook runs
RAW = ROOT / "data/raw/mlb_api"
CLEAN = ROOT / "data/clean"
RAW.mkdir(parents=True, exist_ok=True)

API = "https://statsapi.mlb.com/api/v1"
SEASONS = [2021, 2022, 2023, 2024, 2025]
USE_IL = False  # injured-list data: off unless the team decides otherwise (ethics section)

starts = pd.read_parquet(CLEAN / "starts.parquet")
print(starts.shape)

(18669, 10)


## Game temperature and conditions

One schedule request per season, saved as raw JSON. `hydrate=weather` attaches the game-time weather that MLB records at first pitch.

Domes and closed roofs report a controlled temperature, so the `indoor` flag lets the baseline regression treat them separately. TODO: decide how (indicator, or drop temperature for indoor games).

In [2]:
for year in SEASONS:
    path = RAW / f"schedule_{year}.json"
    if path.exists():
        continue
    r = requests.get(f"{API}/schedule", timeout=120, params={
        "sportId": 1, "gameType": "R", "startDate": f"{year}-03-01", "endDate": f"{year}-11-30",
        "hydrate": "weather,venue"})
    r.raise_for_status()
    path.write_text(r.text)

rows = []
for year in SEASONS:
    for day in json.loads((RAW / f"schedule_{year}.json").read_text())["dates"]:
        for g in day["games"]:
            w = g.get("weather", {})
            rows.append({"game_pk": g["gamePk"], "state": g["status"]["detailedState"],
                         "venue": g["venue"]["name"], "day_night": g.get("dayNight"),
                         "temp_f": w.get("temp"), "condition": w.get("condition"), "wind": w.get("wind")})

games = pd.DataFrame(rows)
games = games[games.state.isin(["Final", "Completed Early"])].drop_duplicates("game_pk", keep="last")
games["temp_f"] = pd.to_numeric(games.temp_f, errors="coerce")
games["indoor"] = games.condition.isin(["Dome", "Roof Closed"])
print(f"{len(games):,} games; temperature missing for "
      f"{games[games.game_pk.isin(starts.game_pk)].temp_f.isna().mean():.1%} of our starts")

12,148 games; temperature missing for 0.0% of our starts


### Fill missing temperatures from the live game feed

Only for our games that the schedule left without a temperature. Each game's weather block is cached as its own small JSON file.

In [3]:
feed_dir = RAW / "game_weather"
feed_dir.mkdir(exist_ok=True)
missing = games.loc[games.temp_f.isna() & games.game_pk.isin(starts.game_pk), "game_pk"]
for pk in missing:
    path = feed_dir / f"{pk}.json"
    if not path.exists():
        r = requests.get(f"https://statsapi.mlb.com/api/v1.1/game/{pk}/feed/live", timeout=60)
        r.raise_for_status()
        path.write_text(json.dumps(r.json()["gameData"].get("weather", {})))

filled = {int(p.stem): json.loads(p.read_text()) for p in feed_dir.glob("*.json")}
fill_temp = games.game_pk.map(lambda pk: filled.get(pk, {}).get("temp"))
games["temp_f"] = games.temp_f.fillna(pd.to_numeric(fill_temp, errors="coerce"))
games = games.drop(columns="state")

## Pitcher information

Birth dates come from the MLB Stats API `people` endpoint (100 ids per request). Names and cross-reference ids come from `pybaseball`, which uses the Chadwick Bureau register.

The cache is a single file; delete `people.json` if the set of pitchers changes.

In [4]:
ids = sorted(starts.pitcher.unique())
path = RAW / "people.json"
if not path.exists():
    people = []
    for i in range(0, len(ids), 100):
        r = requests.get(f"{API}/people", timeout=60,
                         params={"personIds": ",".join(map(str, ids[i:i + 100]))})
        r.raise_for_status()
        people += r.json()["people"]
    path.write_text(json.dumps(people))

pitchers = pd.DataFrame([{"pitcher": p["id"], "name": p["fullName"],
                          "birth_date": p.get("birthDate"), "throws": p["pitchHand"]["code"]}
                         for p in json.loads(path.read_text())])
pitchers["birth_date"] = pd.to_datetime(pitchers.birth_date)

# Untested: needs network the first time (downloads the Chadwick register)
ids_map = playerid_reverse_lookup(ids, key_type="mlbam")[["key_mlbam", "key_fangraphs", "key_bbref"]]
pitchers = pitchers.merge(ids_map.rename(columns={"key_mlbam": "pitcher"}), on="pitcher", how="left")
print(pitchers.shape, "missing birth dates:", pitchers.birth_date.isna().sum())

(316, 6) missing birth dates: 0


## Pitcher type: power vs. finesse (training seasons only)

Does our answer differ by pitcher type? The type is fixed from **2021 to 2023 pitches only**, so test starts never inform it.

- `fb_velo`: mean four-seam or sinker velocity.
- `k_rate`: strikeouts per plate appearance ended by the pitcher.
- `power`: above the training median on both. TODO: settle the definition (for example K + BB rate, or a 2-cluster GMM on these features) and cite it.

Pitchers with no training starts (debuts in 2024 or 2025) get `NaN` here. TODO: decide whether to classify them from their first validation starts, or leave them out of the by-type comparison.

In [5]:
pitches = pd.read_parquet(CLEAN / "statcast_starters.parquet")
train = pitches[pitches.split == "train"]

fb_velo = train[train.pitch_type.isin(["FF", "SI"])].groupby("pitcher").release_speed.mean()
pa = train[train.events.notna()]
k_rate = pa.events.isin(["strikeout", "strikeout_double_play"]).groupby(pa.pitcher).mean()

ptype = pd.concat([fb_velo.rename("fb_velo"), k_rate.rename("k_rate")], axis=1)
ptype["power"] = (ptype.fb_velo > ptype.fb_velo.median()) & (ptype.k_rate > ptype.k_rate.median())
pitchers = pitchers.merge(ptype.reset_index(), on="pitcher", how="left")
pitchers.power.value_counts(dropna=False)

power
False    166
True      77
<NA>      73
Name: count, dtype: Int64

## (Optional) Injured-list transactions

Off by default (`USE_IL = False`). Using injury data raises the ethics questions in the brief (inferring player health from public data), so only turn this on after the team agrees, and discuss it in the Ethics section.

> **Tested on 2023** (1,841 pitcher-level rows matched "injured list"; no other phrasing such as "IL" or "disabled" was missed). Findings that shaped the code:
> - About 43% of matches are *activations* and 10% are list-to-list transfers, not new injuries, so the `event` column separates `placed`, `activated` and `transferred`. Use only `placed` for injury onset.
> - The API repeats some rows (22 in 2023), so duplicates are dropped.
> - Descriptions often name the injury ("Right elbow UCL reconstruction"), which is sensitive health information; keep that text out of any report table.
> - 2021, 2022, 2024 and 2025 have not been checked.

In [6]:
if USE_IL:
    for year in SEASONS:
        path = RAW / f"transactions_{year}.json"
        if not path.exists():
            r = requests.get(f"{API}/transactions", timeout=120, params={
                "sportId": 1, "startDate": f"{year}-01-01", "endDate": f"{year}-12-31"})
            r.raise_for_status()
            path.write_text(r.text)
    tx = pd.DataFrame([{"pitcher": t["person"]["id"], "date": t.get("date"), "description": t.get("description", "")}
                       for y in SEASONS for t in json.loads((RAW / f"transactions_{y}.json").read_text())["transactions"]
                       if "person" in t])
    il = tx[tx.pitcher.isin(ids) & tx.description.str.contains("injured list", case=False)].copy()
    # Each row is a placement, an activation or a transfer between lists (e.g. 15-day to 60-day)
    il["event"] = il.description.str.extract(r"\b(placed|activated|transferred)\b", expand=False)
    il = il.drop_duplicates(["pitcher", "date", "description"])  # the API repeats some rows
    il.to_parquet(CLEAN / "injured_list.parquet", index=False)

## Merge onto starts and pitches, then save

- `age`: years at the game date.
- `temp_f`, `indoor`: from step 1.

Standardizing temperature or age for a model happens in the modeling notebook, using training-season means and standard deviations only.

In [7]:
starts_info = (starts.merge(games, on="game_pk", how="left")
                     .merge(pitchers[["pitcher", "birth_date", "fb_velo", "k_rate", "power"]], on="pitcher", how="left"))
starts_info["age"] = (starts_info.game_date - starts_info.birth_date).dt.days / 365.25

print("Missing share by split:")
display(starts_info.groupby("split")[["temp_f", "age", "power"]].apply(lambda g: g.isna().mean()).round(3))

cols = ["start_id", "temp_f", "indoor", "day_night", "venue", "age", "power"]
model = pitches.merge(starts_info[cols], on="start_id", how="left", validate="many_to_one")
assert len(model) == len(pitches)

games.to_parquet(CLEAN / "games.parquet", index=False)
pitchers.to_parquet(CLEAN / "pitchers.parquet", index=False)
model.to_parquet(CLEAN / "pitches_model.parquet", index=False)
print(model.shape)

Missing share by split:


,temp_f,age,power
split,,,
test,0.0,0.0,0.305
train,0.0,0.0,0.000
val,0.0,0.0,0.242


(1654821, 64)


## Summary of saved dataframes

All files are in `data/clean/`. Row counts for the three new files will print when the notebook is run; the notebook has not been run yet, so shapes are described here, not quoted.

### `games.parquet`: one row per game
Built from the MLB Stats API schedule for regular-season games that finished (`Final` or `Completed Early`), covering all of 2021 to 2025, not only the games in `starts`.
- `game_pk`: join key to the pitch and start tables.
- `venue`, `day_night`: park name and day or night game.
- `temp_f`: game-time temperature in °F, filled from the live game feed when the schedule had none. Still `NaN` for any game neither source covered.
- `condition`, `wind`: raw weather strings from the API.
- `indoor`: `True` when the condition is `Dome` or `Roof Closed`, where temperature is climate-controlled.

### `pitchers.parquet`: one row per starting pitcher
One row for each pitcher id in `starts.parquet`, any season.
- **Identity:** `pitcher` (MLBAM id), `name`, `birth_date`, `throws`, plus `key_fangraphs` and `key_bbref` from the Chadwick register.
- **Pitcher type** (from 2021 to 2023 pitches only): `fb_velo` (mean four-seam or sinker velocity), `k_rate` (strikeouts per plate appearance ended), and `power` (above the training median on both). Pitchers with no training pitches (2024 and 2025 debuts) have `NaN` for all three.

### `pitches_model.parquet`: one row per pitch
The `statcast_etl` pitch table (`statcast_starters.parquet`, all 58 columns, same row count) with start-level context joined on by `start_id`. Six columns are added: `temp_f`, `indoor`, `day_night`, `venue`, `age` (years at the game date) and `power`. This is the main input for the modeling notebooks, with the z-scored features, pitch count, split labels and fatigue covariates (pitch count, times through the order, temperature) in one table.

`injured_list.parquet` is written only if `USE_IL = True`, which it is not by default.